# 03 - Weather Data: Technical Cleaning & Pre-Merge Standardization

## 1. Objective
Following the exploratory analysis in the previous phase, this notebook executes the strict technical formatting required before the meteorological data can be merged with the OMIP financial data.

Instead of writing raw cleaning logic here, we import and execute the robust `clean_weather.py` module developed in our `src.data` pipeline.

**Key Operations Performed by the Module:**
1. **Data Type Coercion:** Enforcing strict numerical types for sensor data and `Int64` for binary/calendar flags to prevent downstream type conflicts.
2. **Temporal Standardization:** Converting UNIX timestamps (sunrise/sunset) to ISO datetime and ensuring the primary `date` index is unique and sorted.
3. **Data Integrity:** Dropping duplicate dates and running validation asserts (`WeatherCleaningError`) to guarantee no null dates exist.

The output will be a fully sanitized dataset saved in the `data/interim/` layer, ready for the financial merge.

In [3]:
import pandas as pd
import sys
import os
from pathlib import Path

# Dynamically configure the project root directory
project_root = Path(os.path.abspath('../../'))
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import the professional cleaning module
from src.data.clean_weather import clean_weather_dataframe
from src.config.paths import WEATHER_CLEAN_FILE

# Define input path (from our previous aggregation step)
input_path = project_root / "data" / "interim" / "national_weather_aggregated.csv"

# 1. Load the raw aggregated dataset
print("Loading raw aggregated weather data...")
df_raw = pd.read_csv(input_path)
print(f"Raw shape: {df_raw.shape}")

# 2. Execute the Cleaning Pipeline
print("\nExecuting Technical Cleaning Pipeline (src.data.clean_weather)...")
try:
    df_clean = clean_weather_dataframe(df_raw)
    print("✅ Technical cleaning successful.")
except Exception as e:
    print(f"❌ Cleaning failed: {e}")

# 3. Export to Interim Layer
WEATHER_CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(WEATHER_CLEAN_FILE, index=False)
print(f"\n✅ Clean dataset exported to: {WEATHER_CLEAN_FILE}")

Loading raw aggregated weather data...
Raw shape: (2275, 21)

Executing Technical Cleaning Pipeline (src.data.clean_weather)...
✅ Technical cleaning successful.

✅ Clean dataset exported to: C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\interim\weather_clean.csv


## 2. Technical Audit & Validation

Before handing this dataset over to the next pipeline stage (Data Merge), we must audit the results of the cleaning script to ensure structural integrity and correct data typing.

In [4]:
# Re-load to verify the exported file's integrity
df_audit = pd.read_csv(WEATHER_CLEAN_FILE)

print("--- WEATHER DATASET TECHNICAL AUDIT ---")
print(f"Total Records: {len(df_audit)}")
print(f"Total Features: {len(df_audit.columns)}")
print(f"Date Range: {df_audit['date'].min()} to {df_audit['date'].max()}\n")

# Check for Duplicates
duplicates = df_audit['date'].duplicated().sum()
print(f"Duplicate Dates: {duplicates} (Expected: 0)")

# Check Data Types (Specifically our target Int64 casts)
print("\nSample Data Types:")
print(df_audit[['date', 'temperature_2m_mean', 'is_national_holiday', 'sunrise']].dtypes)

# Display Null Value Summary (Only showing columns with missing data)
print("\nNull Value Report:")
null_counts = df_audit.isnull().sum()
null_report = null_counts[null_counts > 0]
if len(null_report) == 0:
    print("✅ Zero missing values detected across all columns.")
else:
    print("⚠️ Missing values detected:")
    print(null_report)

--- WEATHER DATASET TECHNICAL AUDIT ---
Total Records: 2275
Total Features: 21
Date Range: 2020-01-01 to 2026-03-24

Duplicate Dates: 0 (Expected: 0)

Sample Data Types:
date                       str
temperature_2m_mean    float64
is_national_holiday      int64
sunrise                    str
dtype: object

Null Value Report:
✅ Zero missing values detected across all columns.


## 3. Anomaly Resolution & Handoff to Merge Phase

**Data Quality Finding: The Null Value Status**
The audit confirms that the core meteorological features and binary flags (`is_national_holiday`) have been successfully cast to their correct data types. Duplicate rows have been eliminated, establishing a strict one-to-one daily timeline.

**Architectural Decision:**
If any null values remain (for example, in the calculated `sunrise`/`sunset` UNIX conversions depending on the API's edge cases), we intentionally do **not** impute them with averages here. Technical scripts should only format data, not invent it. Any necessary interpolation will be handled during the **Feature Engineering** phase, where we have the context of the merged dataset.

**Conclusion:**
The dataset `weather_clean.csv` has passed the technical quality gate. It is identically structured to `omip_clean.csv` in terms of its daily temporal index.

**Next Pipeline Stage:** Execute `03_merge_datasets.ipynb`